*Technical Interview Preparation System (TIPS)*:

A system that analyzes resumes and job descriptions to generate tailored interview questions.

In [ ]:
#SYSTEM IMPORTS:

import os
import sys
import subprocess
import importlib.util
from typing import List, Dict, Optional, TypedDict, Annotated
import json
import uuid
import re
from pathlib import Path
from collections import Counter
from dataclasses import dataclass
import logging
import argparse

In [ ]:
#REQUIREMENTS CHECKER/INSTALLER:

def check_and_install_requirements():
    """Check if required packages are installed, install if missing, and restart if needed."""

    # Define required packages
    requirements = [
        "langchain>=0.1.0",
        "langchain-community>=0.1.0",
        "langchain-openai>=0.1.0",
        "langchain-experimental>=0.1.0",
        "langchain-text-splitters>=0.1.0",
        "langchain-huggingface>=0.1.0",
        "langgraph>=0.1.0",
        "chromadb>=0.4.0",
        "sentence-transformers>=2.2.0",
        "pypdf>=3.0.0",
        "python-docx>=0.8.0",
        "pdfplumber>=0.9.0",
        "beautifulsoup4>=4.11.0",
        "requests>=2.28.0",
        "python-dotenv>=1.0.0",
        "openai>=1.0.0",
        "tiktoken>=0.5.0",
        "tavily-python>=0.3.0",
        "transformers>=4.30.0",
        "reportlab",  # For PDF generation
        "pydantic",  # For data validation
        "faiss-cpu"  # For vector similarity
    ]

    # Create requirements.txt if it doesn't exist
    requirements_file = Path("requirements.txt")
    requirements_file.write_text("\n".join(requirements))

    # Check which packages are missing
    missing_packages = []
    for req in requirements:
        package_name = req.split(">=")[0].split("==")[0]
        package_name_import = package_name.replace("-", "_")

        if importlib.util.find_spec(package_name_import) is None:
            # Special cases for package names that differ from import names
            special_cases = {
                "pypdf": "pypdf",
                "python-docx": "docx",
                "beautifulsoup4": "bs4",
                "python-dotenv": "dotenv",
                "tavily-python": "tavily",
                "faiss-cpu": "faiss"
            }

            actual_import = special_cases.get(package_name, package_name_import)
            if importlib.util.find_spec(actual_import) is None:
                missing_packages.append(req)

    # Install missing packages if any
    if missing_packages:
        print(f"Missing packages detected: {missing_packages}")
        print("Installing missing packages...")

        for package in missing_packages:
            print(f"Installing {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

        # Check if we're in Google Colab
        try:
            import google.colab
            print("\nPackages installed. Restarting runtime...")
            print("Please run the cell again after the restart.")
            #os.kill(os.getpid(), 9)  # Force restart in Colab
        except ImportError:
            print("\nPackages installed successfully.")
            print("You may need to restart the kernel for changes to take effect.")
    else:
        print("All required packages are already installed.")

# Run requirements check
check_and_install_requirements()

All required packages are already installed.


In [ ]:
#THIRD-PARTY IMPORTS:

try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
except ImportError:
    print("PyTorch not found - some features may be limited")

# LangChain and related imports
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.agents import create_openai_functions_agent, AgentExecutor
from langgraph.graph import StateGraph, add_messages, START, END

# PDF Generation Imports
from reportlab.lib.pagesizes import LETTER
from reportlab.pdfgen import canvas
from textwrap import wrap
from pathlib import Path
from datetime import datetime
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer
from reportlab.lib.units import inch

# External API clients
from tavily import TavilyClient
import chromadb

# Configure logging
logging.basicConfig(level=logging.WARNING, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

PyTorch version: 2.6.0+cu124
CUDA available: True


In [ ]:
#PDF GENERATION:

def create_styled_pdf(text: str, out_path: str | Path):
    """
    Creates a styled PDF from a string that has markdown-like cues.
    This version uses ReportLab's Platypus framework for better flow control.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    doc = SimpleDocTemplate(str(out_path), pagesize=LETTER)
    styles = getSampleStyleSheet()

    # Define custom styles
    # Normal text style
    style_normal = styles['BodyText']
    style_normal.fontName = 'Helvetica'
    style_normal.fontSize = 10
    style_normal.leading = 14

    # Main Header (H1) style
    style_h1 = styles['h1']
    style_h1.fontName = 'Helvetica-Bold'
    style_h1.fontSize = 16
    style_h1.spaceAfter = 18

    # Sub-header (H2) style
    style_h2 = styles['h2']
    style_h2.fontName = 'Helvetica-Bold'
    style_h2.fontSize = 12
    style_h2.spaceAfter = 12
    style_h2.spaceBefore = 12

    # Code style
    style_code = styles['Code']
    style_code.fontName = 'Courier'
    style_code.fontSize = 9
    style_code.leading = 12
    style_code.leftIndent = 18
    style_code.wordWrap = 'CJK' # Handles long lines without spaces
    style_code.backColor = '#F0F0F0'
    style_code.borderPadding = 5

    story = []
    in_code_block = False

    for line in text.splitlines():
        # Skip the '========' separators
        if line.strip() == "="*80:
            continue

        # H1 Headers
        if line in ["COMPANY PROFILE", "GENERATED INTERVIEW QUESTIONS AND ANSWERS"]:
            story.append(Paragraph(line, style_h1))
            continue

        # H2 Headers
        if line.startswith("## "):
            story.append(Paragraph(line.lstrip("## "), style_h2))
            continue

        # Code block toggle
        if line.strip().startswith("```"):
            in_code_block = not in_code_block
            if in_code_block:
                code_text = [] # Start collecting code lines
            else:
                # End of a code block, add it to the story
                full_code = "<br/>".join(code_text).replace(' ', ' ')
                story.append(Paragraph(full_code, style_code))
                story.append(Spacer(1, 0.2*inch))
            continue

        if in_code_block:
            code_text.append(line)
        else:
            # Normal paragraph text
            if line.strip(): # Avoid adding empty paragraphs
                 story.append(Paragraph(line, style_normal))
            else:
                 story.append(Spacer(1, 0.2*inch)) # Add a small space for empty lines

    doc.build(story)

In [ ]:
#CONFIG CLASS:

@dataclass
class Config:
    """Central configuration for the application"""
    # Paths
    data_dir: Path = Path("data")
    chroma_dir: Path = Path("chroma_data")

    # Model settings
    embed_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    llm_model: str = "gpt-4o-mini"

    # Chunking parameters
    chunk_size: int = 750
    chunk_overlap: int = 150

    # Retrieval settings
    top_k: int = 5

    # API settings
    temperature: float = 0.3

In [ ]:
#ENVIRONMENT MANAGER CLASS:

class EnvironmentManager:
    """Manages environment setup and API keys"""

    @staticmethod
    def setup_colab():
        """Setup Google Colab environment"""
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=True)
            return True
        except ImportError:
            logger.info("Not running in Google Colab")
            return False

    @staticmethod
    def load_api_keys(folder_path: Optional[str] = None):
        """Load API keys from files or environment variables"""
        if folder_path:
            # Load from files (Colab mode)
            key_files = {
                "OPENAI_API_KEY": "openai_key.txt",
                "TAVILY_API_KEY": "tavily_key.txt"
            }

            for env_var, filename in key_files.items():
                key_path = Path(folder_path) / filename
                if key_path.exists():
                    with open(key_path, "r") as f:
                        os.environ[env_var] = f.read().strip()
                else:
                    logger.warning(f"Key file {filename} not found at {key_path}")

        # Verify keys are loaded
        required_keys = ["OPENAI_API_KEY", "TAVILY_API_KEY"]
        missing_keys = [key for key in required_keys if not os.environ.get(key)]

        if missing_keys:
            raise ValueError(f"Missing required API keys: {missing_keys}")

        logger.info("All API keys loaded successfully")

In [ ]:
#DOCUMENT CLEANER CLASS:

class DocumentCleaner:
    """Handles document cleaning and preprocessing"""

    NOISE_PATTERNS = [
        r"https?://\S+",                    # URLs
        r"\ball rights reserved\b",         # Legal lines
        r"\bapply\b.*\b(now|today)\b",     # Apply links
        r"page\s*\d+\s*(?:of|/)\s*\d+",   # Page counters
        r"^\s*-{2,}\s*$",                 # HR rules
    ]

    def __init__(self):
        self.noise_regex = re.compile("|".join(self.NOISE_PATTERNS), re.I)

    def clean_job_pdf(self, path: str) -> str:
        """Clean job posting PDF by removing noise and repeated content"""
        pages = PyPDFLoader(path).load()
        raw_pages = [p.page_content for p in pages]

        # Detect repeated lines (appearing in >20% of pages)
        line_freq = Counter(
            line.strip().lower()
            for page in raw_pages
            for line in page.splitlines()
            if line.strip()
        )

        repeat_threshold = max(2, int(0.2 * len(raw_pages)))
        repeated_lines = {
            line for line, count in line_freq.items()
            if count >= repeat_threshold
        }

        # Filter and deduplicate
        seen = set()
        cleaned_lines = []

        for page in raw_pages:
            for line in page.splitlines():
                if self._is_noise_line(line, repeated_lines):
                    continue

                normalized = line.strip().lower()
                if normalized not in seen:
                    seen.add(normalized)
                    cleaned_lines.append(line.strip())

        # Join and clean up excessive newlines
        cleaned = re.sub(r"\n{2,}", "\n", "\n".join(cleaned_lines)).strip()
        return cleaned

    def _is_noise_line(self, line: str, repeated: set) -> bool:
        """Check if a line is noise that should be filtered"""
        line_stripped = line.strip()

        if not line_stripped or len(line_stripped) < 5:
            return True

        if line_stripped.lower() in repeated:
            return True

        if self.noise_regex.search(line_stripped):
            return True

        return False

In [ ]:
#VECTOR STORE MANAGER CLASS:

class VectorStoreManager:
    """Manages ChromaDB vector store operations"""

    def __init__(self, config: Config):
        self.config = config
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=config.chunk_size,
            chunk_overlap=config.chunk_overlap
        )
        self.embeddings = HuggingFaceEmbeddings(model_name=config.embed_model)
        self.client = chromadb.PersistentClient(path=str(config.chroma_dir))
        self.cleaner = DocumentCleaner()

    def get_collection(self, name: str):
        """Get or create a ChromaDB collection"""
        return self.client.get_or_create_collection(name)

    def load_and_split(self, path: str) -> List[Document]:
        """Load document and split into chunks"""
        path_obj = Path(path)

        if path_obj.suffix.lower() == ".pdf":
            docs = PyPDFLoader(str(path)).load()
        elif path_obj.suffix.lower() == ".docx":
            docs = Docx2txtLoader(str(path)).load()
        else:
            docs = TextLoader(str(path), encoding="utf-8").load()

        return self.splitter.split_documents(docs)

    def add_chunks(self, chunks: List[Document], collection_name: str, prefix: str):
        """Add document chunks to a collection"""
        collection = self.get_collection(collection_name)

        texts = [chunk.page_content for chunk in chunks]
        ids = [f"{prefix}_{i}_{uuid.uuid4().hex[:6]}" for i in range(len(chunks))]

        collection.add(
            documents=texts,
            metadatas=[chunk.metadata for chunk in chunks],
            ids=ids,
            embeddings=self.embeddings.embed_documents(texts)
        )

    def mmr_query(self, query: str, collection_name: str, k: int = None) -> List[str]:
        """Perform MMR (Maximal Marginal Relevance) search"""
        if k is None:
            k = self.config.top_k

        vector_store = Chroma(
            client=self.client,
            collection_name=collection_name,
            embedding_function=self.embeddings,
            persist_directory=str(self.config.chroma_dir)
        )

        retriever = vector_store.as_retriever(
            search_type="mmr",
            search_kwargs={"k": k}
        )

        docs = retriever.get_relevant_documents(query)
        # Filter out very short documents
        docs = [d for d in docs if len(d.page_content) > 40]

        return [d.page_content for d in docs][:k]

In [ ]:
#TOOLS:

def create_tools(vector_manager: VectorStoreManager, config: Config):
    """Create and return all tools for the agents"""

    @tool
    def resume_ingest(path: str) -> str:
        """Parse résumé and return a JSON profile with top skills & projects."""
        chunks = vector_manager.load_and_split(path)
        vector_manager.add_chunks(chunks, "resumes", "R")
        facts = " ".join(vector_manager.mmr_query("key skills and projects", "resumes"))
        return facts

    @tool
    def company_research(job_pdf_path: str,
                        company_query: str,
                        k_web: int = 5,
                        k_job: int = 8) -> str:
        """Build company profile by mixing cleaned PDF text & Tavily search."""

        # Clean job posting
        if Path(job_pdf_path).suffix.lower() == ".pdf":
            job_text = vector_manager.cleaner.clean_job_pdf(job_pdf_path)
        else:
            job_text = Path(job_pdf_path).read_text(encoding="utf-8", errors="ignore")

        # Add to vector DB
        job_chunks = vector_manager.load_and_split(job_pdf_path)
        vector_manager.add_chunks(job_chunks, "jobs", f"J{uuid.uuid4().hex[:4]}")
        job_facts = " ".join(vector_manager.mmr_query("technical requirements", "jobs", k=k_job))

        # Web search
        tavily = TavilySearchResults(
            k=k_web,
            tavily_api_key=os.environ["TAVILY_API_KEY"]
        )
        web_raw = tavily.run({"query": company_query})
        web_facts = "\n".join(w["content"] for w in web_raw)

        # Generate company profile
        summarize_prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a JSON‐only generator. Output *only* valid JSON, no extra text."),
            ("user", """
              Combine these JOB FACTS and WEB FACTS into a JSON object with keys:
                • stack (list of strings)
                • culture (2–5 sentences)
                • recent_news (list of <5 bullets)
                • mission (string)

            ### JOB FACTS ###
            {job}

            ### WEB FACTS ###
            {web}

            Output *only* the JSON object.
            """),
        ])

        llm = ChatOpenAI(model=config.llm_model, temperature=config.temperature)
        profile_json = (summarize_prompt | llm).invoke({
            "job": job_facts,
            "web": web_facts
        }).content

        try:
            company_profile = json.loads(profile_json)
        except json.JSONDecodeError:
            logger.error(f"Invalid JSON from LLM: {profile_json}")
            raise

        output_dict = {
            "job_facts": job_facts,
            "company_profile": company_profile,
        }

        return json.dumps(output_dict)

    @tool
    def qa_generate(resume_facts: str, company_facts: str) -> str:
        """Generate tailored interview Q&A in Markdown."""
        prompt = PromptTemplate.from_template("""
You are **TIPS** – the technical interview prep system.

**Candidate profile:**
{resume}

**Company profile:**
{company}

Generate:
    - 4 technical interview questions (with their accompanying in-depth answers/explanations). If applicable, these should be coding problems. Your answers will ultimately be returned in a markdown format, so make sure they are easy to read and properly tabbed/indented.
    - 2 behavioral or culture-fit questions (also with their accompanying answers). Add a short disclaimer to these answers explaining that they aren't necessarily true or accurate.

    Return an easy-to-read numbered Markdown list.
""")

        llm = ChatOpenAI(model=config.llm_model, temperature=config.temperature)
        return (prompt | llm).invoke({
            "resume": resume_facts,
            "company": company_facts
        }).content

    return {
        "resume_ingest": resume_ingest,
        "company_research": company_research,
        "qa_generate": qa_generate
    }

In [ ]:
#TIPS PIPELINE CLASS:

# Define state
class State(TypedDict, total=False):
    """Represents the state of the graph."""
    resume_path: str
    job_path: str
    company_query: str

    #intermediate results
    resume_facts: str
    company_profile: dict
    job_facts: str

    #final output
    qa_markdown: str

class InterviewPrepPipeline:
    """Main pipeline for interview preparation"""

    def __init__(self, config: Config, verbose: bool = False):
        self.config = config
        self.verbose = verbose
        self.vector_manager = VectorStoreManager(config)
        self.tools = create_tools(self.vector_manager, config)


        llm = ChatOpenAI(model=self.config.llm_model, temperature=self.config.temperature)
        self.resume_agent = self._build_agent(
            llm,
            [self.tools["resume_ingest"]],
            system_prompt=(
                "You are a Resume Analysis Agent. Your sole purpose is to call the `resume_ingest` tool."
                " After the tool has run, you MUST output the raw text that the tool provides, and nothing else."
            ),
            user_template="{input}"
        )

        self.company_agent = self._build_agent(
            llm,
            [self.tools["company_research"]],
            system_prompt=(
                "You are a Company Research Agent. Your sole purpose is to call the `company_research` tool."
                " After the tool has run, you MUST output the raw JSON string that the tool provides, and nothing else."
            ),
            user_template="{input}"
        )

        self.qa_agent = self._build_agent(
            llm,
            [self.tools["qa_generate"]],
            system_prompt=(
                "You are a Question Generation Agent. Your sole purpose is to call the `qa_generate` tool."
                " After the tool has run, you MUST output the raw Markdown text that the tool provides, and nothing else."
            ),
            user_template="Candidate profile:\n{resume_facts}\n\nCompany profile:\n{company_facts}",
        )

        # Compile the workflow once on initialization
        self.workflow = self._build_workflow()

    def _build_agent(self, llm: ChatOpenAI, tools: list, system_prompt: str, user_template: str):
        """Build an agent executor"""
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("user", user_template),
            MessagesPlaceholder(variable_name="agent_scratchpad"),
        ])
        agent = create_openai_functions_agent(llm=llm, tools=tools, prompt=prompt)
        return AgentExecutor(agent=agent, tools=tools, verbose=False)

    def _run_resume_agent(self, state: State) -> Dict[str, str]:
        """Node to process the resume."""
        if self.verbose: print("--- Running Resume Agent ---")
        agent_output = self.resume_agent.invoke({"input": state["resume_path"]})

        return {"resume_facts": agent_output['output']}

    def _run_company_agent(self, state: State) -> Dict[str, any]:
        """Node to research the company."""
        if self.verbose: print("--- Running Company Research Agent ---")
        agent_output = self.company_agent.invoke({
            "input": json.dumps({
                "job_pdf_path": state["job_path"],
                "company_query": state["company_query"],
            })
        })

        output_data = json.loads(agent_output['output'])

        #access dictionary keys
        company_profile = output_data['company_profile']
        job_facts = output_data['job_facts']

        # print the result
        if self.verbose:
            print("\n--- Generated Company Profile ---")
            print(json.dumps(company_profile, indent=2))
            print("---------------------------------\n")

        return {
            "company_profile": company_profile,
            "job_facts": job_facts,
        }

    def _run_qa_agent(self, state: State) -> Dict[str, str]:
        """Node to generate final Q&A."""
        if self.verbose: print("--- Running QA Generation Agent ---")
        # Combine the company profile and job facts for a comprehensive context
        company_facts_combined = (
            f"**Company Profile & Mission**\n{json.dumps(state['company_profile'], indent=2)}\n\n"
            f"**Key Job Requirements & Tech Stack**\n{state['job_facts']}"
        )

        agent_output = self.qa_agent.invoke({
            "resume_facts": state["resume_facts"],
            "company_facts": company_facts_combined
        })
        return {"qa_markdown": agent_output['output']}

    def _build_workflow(self):
        """Build the LangGraph workflow"""
        graph = StateGraph(State)

        # Add nodes using the corrected wrapper functions
        graph.add_node("resume_node", self._run_resume_agent)
        graph.add_node("company_node", self._run_company_agent)
        graph.add_node("qa_node", self._run_qa_agent)

        # Wire the graph
        graph.add_edge(START, "resume_node")
        graph.add_edge("resume_node", "company_node")
        graph.add_edge("company_node", "qa_node")
        graph.add_edge("qa_node", END)

        return graph.compile()

    def run(self, resume_path: str, job_path: str, company_query: Optional[str] = None) -> str:
        """Execute the pipeline and return generated Q&A"""
        if company_query is None:
            company_query = Path(job_path).stem.replace("_", " ").split(".pdf")[0]

        initial_state = {
            "resume_path": resume_path,
            "job_path": job_path,
            "company_query": company_query,
        }

        final_state = self.workflow.invoke(initial_state)
        return {
        "company_profile": final_state.get("company_profile", {}),
        "qa_markdown": final_state.get("qa_markdown", "No questions generated.")
    }

In [ ]:
#CLI INTERFACE:

def create_parser():
    """Create command line argument parser"""
    parser = argparse.ArgumentParser(
        description="Technical Interview Prep System (TIPS) - Generate tailored interview Q&A",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  # Use first files in default directories:
  python %(prog)s

  # Specify specific files:
  python %(prog)s --resume path/to/resume.pdf --job path/to/job.pdf

  # Specify company name for research:
  python %(prog)s --resume resume.pdf --job job.pdf --company "Google AI Research"
        """
    )

    parser.add_argument(
        "--resume",
        type=str,
        help="Path to resume file (PDF, DOCX, or TXT). If not specified, uses first file in resumes directory."
    )

    parser.add_argument(
        "--job",
        type=str,
        help="Path to job description file (PDF, DOCX, or TXT). If not specified, uses first file in jobs directory."
    )

    parser.add_argument(
        "--company",
        type=str,
        help="Company name for research. If not specified, extracted from job filename."
    )

    parser.add_argument(
        "--output",
        type=str,
        help="Output file path for generated Q&A (markdown format). If not specified, prints to console."
    )

    parser.add_argument(
        "--verbose",
        action="store_true",
        default=False,
        help="Enable verbose output"
    )

    return parser

In [ ]:
def format_final_output(result: Dict) -> str:
    """Formats the pipeline's output dictionary into a single string."""

    profile = result.get("company_profile", {})
    qa = result.get("qa_markdown", "")

    output_lines = []

    # Format the Company Profile
    output_lines.append("="*80)
    output_lines.append("COMPANY PROFILE")
    output_lines.append("="*80)

    if profile:
        output_lines.append(f"\n## Mission\n{profile.get('mission', 'Not available.')}\n")
        output_lines.append(f"## Culture\n{profile.get('culture', 'Not available.')}\n")

        stack = profile.get('stack', [])
        if stack:
            output_lines.append("## Technical Stack")
            for item in stack:
                output_lines.append(f"- {item}")
            output_lines.append("")

        news = profile.get('recent_news', [])
        if news:
            output_lines.append("## Recent News")
            for item in news:
                output_lines.append(f"- {item}")
            output_lines.append("")
    else:
        output_lines.append("No company profile was generated.\n")

    # Add the Interview Questions
    output_lines.append("="*80)
    output_lines.append("GENERATED INTERVIEW QUESTIONS AND ANSWERS")
    output_lines.append("="*80)
    output_lines.append(f"\n{qa}")

    return "\n".join(output_lines)

In [ ]:
def main(args=None):
    """Main entry point with CLI support"""
    parser = create_parser()
    args = parser.parse_args(args)
    if args.verbose:
      logging.getLogger().setLevel(logging.DEBUG)
      import langchain
      langchain.debug = True
    else:
      logging.getLogger().setLevel(logging.WARNING)
    is_colab = EnvironmentManager.setup_colab()
    if is_colab:
        folder_path = "/content/drive/Shared drives/ECEN523 Final Project Group/Final Project" #Professor - this folder should be shared with you.
        EnvironmentManager.load_api_keys(folder_path)
        default_jobs_path = os.path.join(folder_path, 'data', 'jobs')
        default_resumes_path = os.path.join(folder_path, 'data', 'resumes')
    else:
        folder_path = Path(".")
        EnvironmentManager.load_api_keys()
        default_jobs_path = folder_path / 'data' / 'jobs'
        default_resumes_path = folder_path / 'data' / 'resumes'
    if args.resume:
        resume_path = args.resume
    else:
        resume_files = list(Path(default_resumes_path).glob("*"))
        if not resume_files:
            raise ValueError(f"No resume files found in {default_resumes_path}")
        resume_path = str(resume_files[0]) #INSERT YOUR RESUME IN DATA/RESUMES
    if args.job:
        job_path = args.job
    else:
        job_files = list(Path(default_jobs_path).glob("*"))
        if not job_files:
            raise ValueError(f"No job files found in {default_jobs_path}")
        job_path = str(job_files[0]) #INSERT YOUR JOB IN DATA/JOBS
    if not args.verbose:
      print(f"Processing resume: {Path(resume_path).name}")
      print(f"Processing job: {Path(job_path).name}")
      print("Generating interview materials...")
    else:
      logger.info(f"Processing resume: {Path(resume_path).name}")
      logger.info(f"Processing job: {Path(job_path).name}")
      logger.info("Generating interview materials...")
    config = Config()
    pipeline = InterviewPrepPipeline(config, verbose=args.verbose)
    # --- End of existing main function ---

    # The pipeline now returns a dictionary
    result_dict = pipeline.run(
        resume_path=resume_path,
        job_path=job_path,
        company_query=args.company
    )

    # Use the new helper function to create a single, formatted string
    final_output = format_final_output(result_dict)

    # Handle output
    if args.output:
        # Save to plain-text file
        output_path = Path(args.output)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(final_output)
        print(f"\nResults saved to: {output_path}")
        logger.info(f"Results saved to: {output_path}")
    else:
        # Print to console
        print("\n" + final_output)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    if is_colab:
        pdf_dir = Path("/content/drive/Shared drives/ECEN523 Final Project Group/Final Project/PDFs")
    elif args.output:
        pdf_dir = Path(args.output).parent
    else:
        pdf_dir = Path("output")

    pdf_dir.mkdir(parents=True, exist_ok=True)
    pdf_path = pdf_dir / f"interview_report_{timestamp}.pdf"

    create_styled_pdf(final_output, pdf_path)
    print(f"PDF written to: {pdf_path}")

    return final_output

In [ ]:
# ENTRY POINT:

if __name__ == "__main__":
    # Check if running in interactive environment
    import sys
    if hasattr(sys, 'ps1') or 'ipykernel' in sys.modules:
        result = main([])
    else:
        # Running as script
        result = main()

Mounted at /content/drive
Processing resume: Emma Berry Resume.pdf
Processing job: Deep_Learning_Architect_New_College_Grad_2025.pdf
Generating interview materials...


<ipython-input-8-7909238>:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(model_name=config.embed_model)
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.wa


COMPANY PROFILE

## Mission
NVIDIA's mission is to pioneer accelerated computing to tackle challenges that no one else can solve, transforming industries through innovative technology.

## Culture
NVIDIA is committed to fostering a diverse work environment and is proud to be an equal opportunity employer. We value creativity and autonomy, seeking individuals who are passionate about technology and innovation. Our team is comprised of some of the most brilliant and talented people in the industry, working collaboratively to push the boundaries of what's possible in accelerated computing.

## Technical Stack
- C
- C++
- Perl
- Python
- CUDA
- OpenCL
- MPI
- OpenMP

## Recent News
- NVIDIA is hiring a Deep Learning Architect - New College Grad 2025.
- The estimated salary for this position ranges from $120,000 to $235,750.
- NVIDIA is at the forefront of AI and digital twins technology.
- The company values diversity and does not discriminate in hiring practices.

GENERATED INTERVIEW QUE